# WDMamba: đánh giá độc lập trên Google Colab

**Chọn Runtime → Change runtime type → GPU**, rồi chạy các cell theo thứ tự.
Notebook tự clone `Darkin72/WDMamba`, tạo môi trường Python 3.11/CUDA,
tải SOTS và trọng số. Không cần repo `VDT_GD2` hoặc chạy `Bench.ipynb` trước.

- Chỉ đánh giá **SOTS-ITS (indoor)** và **SOTS-OTS (outdoor)**.
- **Batch size 16**, không tự động giảm batch khi thiếu VRAM. Batch cuối có thể nhỏ hơn.
- Cả hai tập dùng trọng số `reside6k_32.15.pth`: đây là đánh giá chuyển miền.
- Đo DWT, LFRN, IWT, DEN; đo riêng forward gốc để tính FPS không bị tác động bởi instrumentation.
- Báo cáo PSNR/SSIM, thời gian mỗi batch/mỗi ảnh, FPS và VRAM.
- Báo cáo GFLOPs đếm được và các phần ước lượng bổ sung cho selective scan/FFT/wavelet.

Mặc định giữ tỉ lệ ảnh, giới hạn cạnh dài ở **512 pixel**, giống cấu hình trong `Bench.ipynb`.
Đặt `MAX_SIDE = 0` nếu cần điểm số ở độ phân giải gốc. Có thêm phép đếm tham chiếu 256 × 256.


## 1. Clone repository và cấu hình

Notebook có thể được mở trực tiếp từ GitHub hoặc tải lên Colab. Nếu repo đã tồn tại,
cell này tái sử dụng checkout; không reset hay ghi đè thay đổi trong repo.


In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

REPO_URL = "https://github.com/Darkin72/WDMamba.git"
REPO_ROOT = Path("/content/WDMamba")
WORK_ROOT = Path("/content/wdmamba_benchmark")
OUTPUT_ROOT = WORK_ROOT / "results"
ENV_ROOT = Path("/content/wdmamba_python311")
PYTHON = ENV_ROOT / "bin/python"

def run_command(args, cwd=None):
    args = [str(arg) for arg in args]
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True,
                          encoding="utf-8", errors="replace", bufsize=1) as proc:
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
            return_code = proc.wait()
        except BaseException:
            proc.kill()
            proc.wait()
            raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, args)

if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("Notebook này dành cho Google Colab với GPU CUDA.")
if not REPO_ROOT.exists():
    run_command(["git", "clone", REPO_URL, REPO_ROOT])
if not (REPO_ROOT / "basicsr/archs/wavemamba_arch.py").is_file():
    raise FileNotFoundError("Checkout không có kiến trúc WDMamba.")
run_command(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT)

BATCH_SIZE = 16
MAX_SIDE = 512           # 0 = độ phân giải gốc; không thay đổi tỉ lệ ảnh.
LIMIT = 0               # 0 = toàn bộ mỗi split; thử 32 ảnh để kiểm tra nhanh.
WARMUP = 3
TIMING_REPEATS = 3
NUM_WORKERS = 2
PROFILE_SIZE = 256
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = {
    "repo_root": str(REPO_ROOT), "work_root": str(WORK_ROOT),
    "output_root": str(OUTPUT_ROOT), "batch_size": BATCH_SIZE,
    "max_side": MAX_SIDE, "limit": LIMIT, "warmup": WARMUP,
    "timing_repeats": TIMING_REPEATS, "num_workers": NUM_WORKERS,
    "profile_size": PROFILE_SIZE,
}
CONFIG_PATH = WORK_ROOT / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Cấu hình:", CONFIG)


## 2. Môi trường CUDA riêng, không cần đổi kernel Colab

Sử dụng cấu hình đã dùng trong `Bench.ipynb`: Python 3.11, PyTorch 2.6.0/CUDA 12.4,
torchvision 0.21.0 và wheel Mamba tương ứng. Các cell sau chạy bằng Python của môi trường này.
`setuptools<81` giữ tương thích với mã wavelet dùng `pkg_resources`.


In [ ]:
run_command(["nvidia-smi"])
run_command([sys.executable, "-m", "pip", "install", "-q", "uv"])
run_command([sys.executable, "-m", "uv", "python", "install", "3.11"])
if not PYTHON.is_file():
    run_command([sys.executable, "-m", "uv", "venv", "--python", "3.11", ENV_ROOT])
run_command([
    sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])
constraints = ENV_ROOT / "constraints.txt"
constraints.write_text("torch==2.6.0\ntorchvision==0.21.0\nnumpy<2\nsetuptools<81\n",
                       encoding="utf-8")
run_command([
    sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
    "--constraint", constraints,
    "numpy<2", "opencv-python<4.12", "scipy", "scikit-image", "pandas",
    "einops", "timm", "pytorch-wavelets", "PyWavelets", "setuptools<81",
    "transformers==4.49.0", "gdown", "kagglehub", "thop", "tqdm",
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl",
    "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl",
])

def run_python(label, python_source):
    script = WORK_ROOT / f"{label}.py"
    import textwrap
    script.write_text(textwrap.dedent(python_source).strip() + "\n", encoding="utf-8")
    run_command([PYTHON, "-u", script, CONFIG_PATH], cwd=REPO_ROOT)

run_python("check_cuda", """
import torch
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
assert torch.cuda.is_available(), 'Không có GPU CUDA trong runtime.'
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
with torch.inference_mode():
    u = torch.zeros(1, 4, 16, device='cuda')
    bc = torch.ones(1, 16, 16, device='cuda')
    y = selective_scan_fn(u, torch.ones_like(u), -bc[0, :4], bc, bc,
                          delta_softplus=True)
    assert torch.isfinite(y).all(), 'Selective scan tạo giá trị không hữu hạn.'
print('Kiểm tra kernel selective scan: thành công.')
""")


## 3. Tải dataset SOTS

Chỉ tải SOTS gồm hai split indoor/outdoor; không tải các tập huấn luyện ITS/OTS.
KaggleHub tái sử dụng cache. Nếu Kaggle yêu cầu đăng nhập, dùng cơ chế xác thực của KaggleHub.


In [ ]:
run_python("download_sots", """
import json, sys
from pathlib import Path
import kagglehub
cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
downloaded = Path(kagglehub.dataset_download(
    'balraj98/synthetic-objective-testing-set-sots-reside'))
roots = {}
for label, domain in [('SOTS-ITS', 'indoor'), ('SOTS-OTS', 'outdoor')]:
    candidates = [downloaded / domain, *downloaded.rglob(domain)]
    matches = [p for p in candidates
               if (p / 'hazy').is_dir() and (p / 'clear').is_dir()]
    if not matches:
        raise FileNotFoundError(f'Không tìm thấy {domain}/hazy và clear dưới {downloaded}')
    roots[label] = str(matches[0].resolve())
    print(label, roots[label])
manifest_path = Path(cfg['work_root']) / 'inputs.json'
data = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
data['datasets'] = roots
manifest_path.write_text(json.dumps(data, indent=2), encoding='utf-8')
""")


## 4. Tải và giải nén trọng số

Dùng cùng archive Google Drive với `Bench.ipynb`. Chỉ công bố tệp ZIP sau khi kiểm tra tính toàn vẹn.
Không cần mount Google Drive; có thể nhập `LOCAL_ZIP` nếu đã có archive trên Drive được mount trước đó.


In [ ]:
LOCAL_ZIP = ""   # Ví dụ: /content/drive/MyDrive/WDMamba_ckpts.zip
CONFIG["local_zip"] = LOCAL_ZIP
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
run_python("download_weights", """
import json, sys, shutil, zipfile
from pathlib import Path
import gdown
cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
root = Path(cfg['work_root']) / 'checkpoints'
root.mkdir(parents=True, exist_ok=True)
target = root / 'WDMamba_ckpts.zip'
if not target.is_file() or not zipfile.is_zipfile(target):
    partial = target.with_suffix('.zip.partial')
    if cfg.get('local_zip'):
        shutil.copy2(cfg['local_zip'], partial)
    else:
        result = gdown.download(id='1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF',
            output=str(partial), quiet=False, use_cookies=False, resume=True)
        if result is None:
            raise RuntimeError('Google Drive không trả về trọng số. Kiểm tra chia sẻ/quota hoặc LOCAL_ZIP.')
    with zipfile.ZipFile(partial) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('Archive trọng số bị lỗi.')
    partial.replace(target)
extract_root = root / 'unzipped'
matches = list(extract_root.rglob('reside6k_32.15.pth')) if extract_root.exists() else []
if not matches:
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(target) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('Archive trọng số bị lỗi.')
        for member in archive.infolist():
            resolved = (extract_root / member.filename).resolve()
            if not resolved.is_relative_to(extract_root.resolve()):
                raise ValueError('Archive chứa đường dẫn ngoài thư mục giải nén.')
        archive.extractall(extract_root)
    matches = list(extract_root.rglob('reside6k_32.15.pth'))
if len(matches) != 1:
    raise FileNotFoundError(f'Cần đúng một reside6k_32.15.pth, tìm thấy: {matches}')
manifest_path = Path(cfg['work_root']) / 'inputs.json'
data = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
data['checkpoint'] = str(matches[0].resolve())
manifest_path.write_text(json.dumps(data, indent=2), encoding='utf-8')
print('Trọng số:', matches[0])
""")


## 5. Định nghĩa benchmark độc lập

Cell dưới tạo mã chạy trong môi trường Python 3.11. Import trực tiếp kiến trúc từ repo đã clone,
không dùng các helper của repo khác và không sửa file kiến trúc.

Đo thời gian stage bằng CUDA events trên cùng stream, đồng bộ một lần sau forward.
FPS dùng forward gốc trong lượt riêng. Kiểm tra hai luồng cho output tương đương trước khi chạy.

GFLOPs bao gồm phần `thop` đếm được và ước lượng bổ sung cho phép chiếu einsum,
selective scan, FFT, DWT/IWT. Công thức bổ sung là xấp xỉ số phép tính; không phải số FLOPs chính xác
của kernel GPU. Các phép normalization/activation/cấp phát/copy không được bao phủ đầy đủ.


In [ ]:
import textwrap
RUNNER_SOURCE = textwrap.dedent(r'''
import ast
import importlib.machinery
import importlib.util
import json
import math
import re
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from thop import profile
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

CFG = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
INPUTS = json.loads((Path(CFG["work_root"]) / "inputs.json").read_text(encoding="utf-8"))
ROOT = Path(CFG["repo_root"])
OUTPUT = Path(CFG["output_root"])
assert CFG["batch_size"] == 16, "Giữ batch size 16 theo cấu hình benchmark."
assert CFG["warmup"] >= 1 and CFG["timing_repeats"] >= 1
assert torch.cuda.is_available(), "Cần GPU CUDA."
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.manual_seed(666)
cv2.setNumThreads(0)

def import_architecture():
    # Tránh __init__ tự động import toàn bộ training stack.
    for name, directory in (
        ("basicsr", ROOT / "basicsr"),
        ("basicsr.utils", ROOT / "basicsr/utils"),
        ("basicsr.archs", ROOT / "basicsr/archs"),
    ):
        if name in sys.modules:
            raise RuntimeError(f"Package đã được import từ trước: {name}")
        spec = importlib.machinery.ModuleSpec(name, loader=None, is_package=True)
        spec.submodule_search_locations = [str(directory)]
        sys.modules[name] = importlib.util.module_from_spec(spec)
    from basicsr.archs.wavemamba_arch import WaveMamba, SS2D
    from basicsr.archs.detail_enhance_net import DENet, fft_bench
    from basicsr.archs.wavelet import DWT, IWT
    from pytorch_wavelets import DWTForward
    return WaveMamba, SS2D, DENet, fft_bench, DWT, IWT, DWTForward

WaveMamba, SS2D, DENet, FFTBench, DWT, IWT, DWTForward = import_architecture()

def load_model():
    state = torch.load(INPUTS["checkpoint"], map_location="cpu", weights_only=False)
    if isinstance(state, dict):
        for key in ("params", "state_dict", "model"):
            if key in state:
                state = state[key]
                break
    state = {key.removeprefix("module."): value for key, value in state.items()}
    groups = {i: set() for i in (1, 2, 3)}
    for key in state:
        match = re.fullmatch(r"restoration_network\.DE\.g([123])\.gp\.(\d+)\.conv1\.weight", key)
        if match:
            groups[int(match[1])].add(int(match[2]))
    counts = {len(v) for v in groups.values()}
    if len(counts) != 1 or next(iter(counts)) not in (4, 6):
        raise ValueError(f"Không nhận diện được số block DEN: {groups}")
    blocks = next(iter(counts))
    assert all(v == set(range(blocks)) for v in groups.values())
    model = WaveMamba(in_chn=3, wf=16, n_l_blocks=[1, 2, 2, 4], ffn_scale=2.0)
    if blocks == 4:
        model.restoration_network.DE = DENet(3, 4)
    model.load_state_dict(state, strict=True)
    return model.cuda().eval(), blocks

EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
def files(root):
    return sorted(p for p in root.iterdir() if p.is_file() and p.suffix.lower() in EXT)

def collect_pairs(root):
    clear = {}
    for path in files(root / "clear"):
        if path.stem in clear:
            raise ValueError(f"Trùng tên ảnh sạch: {path.stem}")
        clear[path.stem] = path
    pairs = []
    for hazy in files(root / "hazy"):
        gt = clear.get(hazy.stem, clear.get(hazy.stem.split("_")[0]))
        if gt is None:
            raise FileNotFoundError(f"Không có ảnh sạch tương ứng với {hazy}")
        pairs.append((hazy.stem, hazy, gt))
    if CFG["limit"] > 0:
        pairs = pairs[:CFG["limit"]]
    if not pairs:
        raise ValueError(f"Không có cặp ảnh dưới {root}")
    return pairs

class PairedDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, index):
        name, hazy_path, gt_path = self.pairs[index]
        hazy, gt = [cv2.imread(str(p), cv2.IMREAD_COLOR) for p in (hazy_path, gt_path)]
        if hazy is None or gt is None:
            raise RuntimeError(f"Không đọc được cặp ảnh {name}")
        hazy, gt = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in (hazy, gt)]
        if hazy.shape[:2] != gt.shape[:2]:
            hazy = cv2.resize(hazy, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_AREA)
        h, w = gt.shape[:2]
        side = CFG["max_side"]
        scale = min(1.0, side / max(h, w)) if side > 0 else 1.0
        if scale < 1:
            size = (max(1, round(w * scale)), max(1, round(h * scale)))
            hazy, gt = [cv2.resize(img, size, interpolation=cv2.INTER_AREA) for img in (hazy, gt)]
        tensor = torch.from_numpy(hazy.copy()).permute(2, 0, 1).float().div_(255)
        return tensor, gt, name

def collate(samples):
    shapes = [item[0].shape[-2:] for item in samples]
    h = math.ceil(max(s[0] for s in shapes) / 32) * 32
    w = math.ceil(max(s[1] for s in shapes) / 32) * 32
    tensors = []
    for x, _, _ in samples:
        dh, dw = h - x.shape[-2], w - x.shape[-1]
        mode = "reflect" if dh < x.shape[-2] and dw < x.shape[-1] else "replicate"
        tensors.append(F.pad(x, (0, dw, 0, dh), mode=mode))
    return torch.stack(tensors), [s[1] for s in samples], [s[2] for s in samples], shapes

class LFRNOnly(nn.Module):
    def __init__(self, restoration):
        super().__init__()
        self.r = restoration
    def forward(self, ll):
        n = self.r
        x0 = n.down_group0(n.layer0(ll))
        x1 = n.down_group1(n.layer1(x0))
        x2 = n.down_group2(n.layer2(x1))
        x3 = n.down_group3(n.layer3(x2))
        x4 = n.down_group4(n.layer4(x3))
        y3 = n.up_group4(n.layer_3(x4, x3))
        y2 = n.up_group3(n.layer_2(y3, x2))
        y1 = n.up_group2(n.layer_1(y2, x1))
        y0 = n.up_group1(n.layer_0(y1, x0))
        return n.last(y0) + ll

def staged_forward(model, x, dwt, lfrn, iwt):
    # Events trên cùng stream; không synchronize giữa các giai đoạn.
    events = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
    events[0].record()
    coefficients = dwt(x)
    events[1].record()
    count = x.shape[0]
    ll = lfrn(coefficients[:count])
    events[2].record()
    coarse = iwt(torch.cat((ll, coefficients[count:]), dim=0))
    events[3].record()
    out = model.restoration_network.DE(coarse)
    events[4].record()
    events[-1].synchronize()
    stages = {name: events[i].elapsed_time(events[i + 1])
              for i, name in enumerate(("DWT", "LFRN", "IWT", "DEN"))}
    return out, stages

def original_forward(model, x):
    start, end = [torch.cuda.Event(enable_timing=True) for _ in range(2)]
    wall = time.perf_counter()
    start.record()
    out = model(x)
    end.record()
    end.synchronize()
    return out, start.elapsed_time(end), 1000 * (time.perf_counter() - wall)

def complexity(module, example, stage):
    # Chỉ cộng phần thop bỏ sót; Conv/Linear đã được thop đếm riêng.
    extras = defaultdict(float)
    handles = []
    original_training = {m: m.training for m in module.modules()}
    original_hooks = {m: set(m._forward_hooks) for m in module.modules()}
    original_buffers = {m: set(m._buffers) for m in module.modules()}

    def extra_hook(m, args, output):
        x = args[0]
        if isinstance(m, SS2D):
            b, h, w, _ = x.shape
            length, d, n, rank, k = h * w, m.d_inner, m.d_state, m.dt_rank, 4
            extras["einsum"] += 2 * b * k * length * d * (rank + 2 * n + rank)
            # Recurrence xấp xỉ, coi exp là một phép; không phản ánh chi phí kernel.
            extras["selective_scan"] += b * k * length * d * (8 * n + 2)
        elif isinstance(m, FFTBench):
            b, c, h, w = x.shape
            # Hai FFT theo upper-style complex FFT estimate; rFFT có thể ít phép hơn.
            extras["fft_pair"] += 10 * b * c * h * w * math.log2(h * w)
        elif isinstance(m, DWTForward):
            extras["internal_haar"] += 4 * x.numel()

    for m in module.modules():
        if isinstance(m, (SS2D, FFTBench, DWTForward)):
            handles.append(m.register_forward_hook(extra_hook))
    try:
        with torch.inference_mode():
            macs, _ = profile(module, inputs=(example,), verbose=False)
    finally:
        for handle in handles:
            handle.remove()
        # thop có thể để lại hook/buffer nếu profiling thất bại.
        for m in module.modules():
            m.training = original_training[m]
            for key in set(m._forward_hooks) - original_hooks[m]:
                del m._forward_hooks[key]
            for key in set(m._buffers) - original_buffers[m]:
                del m._buffers[key]
    counted = 2 * float(macs)
    supplemental = sum(extras.values())
    return {"stage": stage, "counted_GFLOPs": counted / 1e9,
            "supplemental_estimate_GFLOPs": supplemental / 1e9,
            "estimated_GFLOPs_per_image": (counted + supplemental) / 1e9,
            "supplemental_details": dict(extras)}

def shape_complexity(model, h, w):
    x = torch.zeros(1, 3, h, w, device="cuda")
    ll = torch.zeros(1, 3, h // 2, w // 2, device="cuda")
    rows = []
    transform = 4 * x.numel() / 1e9  # Scaling + ba phép cộng/trừ trên mỗi hệ số.
    for stage in ("DWT", "IWT"):
        rows.append({"stage": stage, "counted_GFLOPs": 0.0,
                     "supplemental_estimate_GFLOPs": transform,
                     "estimated_GFLOPs_per_image": transform,
                     "supplemental_details": {"haar": transform * 1e9}})
    rows.append(complexity(LFRNOnly(model.restoration_network).eval(), ll, "LFRN"))
    rows.append(complexity(model.restoration_network.DE, x, "DEN"))
    rows.sort(key=lambda row: ("DWT", "LFRN", "IWT", "DEN").index(row["stage"]))
    for row in rows:
        row.update({"height": h, "width": w,
                    "estimated_GFLOPs_batch16": 16 * row["estimated_GFLOPs_per_image"]})
    return rows

def benchmark_dataset(model, label, root, flop_cache):
    dwt, lfrn, iwt = DWT().eval(), LFRNOnly(model.restoration_network).eval(), IWT().eval()
    pairs = collect_pairs(root)
    loader = DataLoader(PairedDataset(pairs), batch_size=16, shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=True, collate_fn=collate,
        persistent_workers=CFG["num_workers"] > 0, drop_last=False)
    first = next(iter(loader))[0].cuda()
    warm_shapes = set()
    batch_rows, image_rows = [], []

    with torch.inference_mode():
        for _ in range(CFG["warmup"]):
            model(first)
        torch.cuda.synchronize()
        reference = model(first)
        staged, _ = staged_forward(model, first, dwt, lfrn, iwt)
        torch.testing.assert_close(staged, reference, rtol=1e-5, atol=1e-5)
    del first, reference, staged
    print(label, ": output phân rã khớp forward gốc.", flush=True)

    iterator = iter(loader)
    for index in tqdm(range(len(loader)), desc=label):
        pipeline_start = time.perf_counter()
        cpu, gt, names, shapes = next(iterator)
        wait_ms = 1000 * (time.perf_counter() - pipeline_start)
        t = time.perf_counter()
        x = cpu.cuda(non_blocking=True)
        torch.cuda.synchronize()
        h2d_ms = 1000 * (time.perf_counter() - t)
        b, _, h, w = x.shape
        key = (b, h, w)
        if key not in warm_shapes:
            with torch.inference_mode():
                for _ in range(CFG["warmup"]):
                    model(x)
            torch.cuda.synchronize()
            warm_shapes.add(key)
        torch.cuda.reset_peak_memory_stats()
        with torch.inference_mode():
            times, walls, stages = [], [], []
            for _ in range(CFG["timing_repeats"]):
                out, gpu_ms, wall_ms = original_forward(model, x)
                times.append(gpu_ms)
                walls.append(wall_ms)
                if len(times) < CFG["timing_repeats"]:
                    del out
            t = time.perf_counter()
            outputs = (out.clamp(0, 1).permute(0, 2, 3, 1).mul(255).round()
                       .to(torch.uint8).cpu().numpy())
            d2h_ms = 1000 * (time.perf_counter() - t)
            del out
            for _ in range(CFG["timing_repeats"]):
                stage_out, stage_times = staged_forward(model, x, dwt, lfrn, iwt)
                stages.append(stage_times)
                del stage_out
        peak = torch.cuda.max_memory_allocated() / 1024 ** 3
        peak_reserved = torch.cuda.max_memory_reserved() / 1024 ** 3
        row = {"dataset": label, "batch_index": index, "actual_batch_size": b,
               "padded_height": h, "padded_width": w,
               "original_forward_gpu_ms": float(np.mean(times)),
               "original_forward_wall_ms": float(np.mean(walls)),
               "dataloader_wait_ms": wait_ms, "h2d_ms": h2d_ms, "d2h_ms": d2h_ms,
               "peak_allocated_GiB": peak, "peak_reserved_GiB": peak_reserved}
        for name in ("DWT", "LFRN", "IWT", "DEN"):
            row[name + "_ms"] = float(np.mean([s[name] for s in stages]))
        row["stage_sum_ms"] = sum(row[n + "_ms"] for n in ("DWT", "LFRN", "IWT", "DEN"))
        for name, shape, pred, target in zip(names, shapes, outputs, gt):
            pred = pred[:shape[0], :shape[1]]
            if min(pred.shape[:2]) < 11:
                raise ValueError("Ảnh quá nhỏ để tính SSIM window 11.")
            image_rows.append({"dataset": label, "image_id": name,
                "height": shape[0], "width": shape[1],
                "PSNR": float(peak_signal_noise_ratio(target, pred, data_range=255)),
                "SSIM": float(structural_similarity(target, pred, channel_axis=-1,
                    data_range=255, gaussian_weights=True, sigma=1.5,
                    use_sample_covariance=False))})
        if (h, w) not in flop_cache:
            flop_cache[(h, w)] = shape_complexity(model, h, w)
        row["estimated_GFLOPs_per_image"] = sum(
            r["estimated_GFLOPs_per_image"] for r in flop_cache[(h, w)])
        row["estimated_GFLOPs_actual_batch"] = b * row["estimated_GFLOPs_per_image"]
        batch_rows.append(row)
        # Lưu tiến độ sau từng batch để không mất kết quả khi Colab ngắt.
        pd.DataFrame(batch_rows).to_csv(OUTPUT / f"{label}_batches.csv", index=False)
        pd.DataFrame(image_rows).to_csv(OUTPUT / f"{label}_images.csv", index=False)
        del x, cpu, outputs

    batches, images = pd.DataFrame(batch_rows), pd.DataFrame(image_rows)
    total = len(images)
    gpu_ms = batches.original_forward_gpu_ms.sum()
    wall_ms = batches.original_forward_wall_ms.sum()
    summary = {"dataset": label, "images": total, "configured_batch_size": 16,
        "last_batch_size": int(batches.actual_batch_size.iloc[-1]),
        "max_side": CFG["max_side"], "PSNR": float(images.PSNR.mean()),
        "SSIM": float(images.SSIM.mean()), "model_gpu_FPS": total * 1000 / gpu_ms,
        "model_wall_FPS": total * 1000 / wall_ms,
        "original_forward_ms_per_image": gpu_ms / total,
        "original_forward_median_ms_per_batch": float(batches.original_forward_gpu_ms.median()),
        "original_forward_p95_ms_per_batch": float(batches.original_forward_gpu_ms.quantile(.95)),
        "peak_allocated_GiB": float(batches.peak_allocated_GiB.max()),
        "peak_reserved_GiB": float(batches.peak_reserved_GiB.max()),
        "estimated_GFLOPs_mean_per_image": float(
            (batches.estimated_GFLOPs_per_image * batches.actual_batch_size).sum() / total)}
    for name in ("DWT", "LFRN", "IWT", "DEN"):
        summary[name + "_ms_per_image"] = float(batches[name + "_ms"].sum() / total)
    return summary

def main():
    model, blocks = load_model()
    print('GPU:', torch.cuda.get_device_name(0), '| Precision: FP32 | DEN:', blocks, flush=True)
    metadata = {"config": CFG, "checkpoint": INPUTS["checkpoint"],
        "protocol": "RESIDE-6K checkpoint -> SOTS transfer evaluation",
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0), "precision": "FP32; TF32 disabled",
        "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
        "LFRN_params_M": sum(p.numel() for n, p in model.restoration_network.named_parameters()
                                if not n.startswith("DE.")) / 1e6,
        "DEN_params_M": sum(p.numel() for p in model.restoration_network.DE.parameters()) / 1e6,
        "flops_note": "thop 2*MACs + analytic scan/FFT/wavelet estimates; incomplete elementwise coverage",
        "metric_note": "RGB uint8; Gaussian SSIM sigma=1.5, population covariance, no crop border"}
    flop_cache = {}
    size = CFG["profile_size"]
    assert size >= 32 and size % 32 == 0
    flop_cache[(size, size)] = shape_complexity(model, size, size)
    summaries = []
    try:
        for label in ("SOTS-ITS", "SOTS-OTS"):
            summaries.append(benchmark_dataset(model, label, Path(INPUTS["datasets"][label]), flop_cache))
            pd.DataFrame(summaries).to_csv(OUTPUT / "summary.csv", index=False)
    finally:
        rows = [row for items in flop_cache.values() for row in items]
        pd.DataFrame(rows).to_csv(OUTPUT / "stage_gflops.csv", index=False)
        (OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    print(pd.DataFrame(summaries).to_string(index=False), flush=True)
    print('Kết quả:', OUTPUT, flush=True)

if __name__ == "__main__":
    try:
        main()
    except torch.cuda.OutOfMemoryError:
        print('Thiếu VRAM ở batch 16. Giữ batch 16, giảm MAX_SIDE hoặc chọn GPU có VRAM lớn hơn.', flush=True)
        raise
''').strip() + "\n"
# Kiểm tra cú pháp ngay trong kernel notebook; không import torch từ môi trường khác.
import ast
ast.parse(RUNNER_SOURCE)
print("Đã tạo mã benchmark độc lập.")


## 6. Chạy benchmark hai split

Đặt `LIMIT = 32` ở cell cấu hình để thử trước; dùng `LIMIT = 0` cho toàn bộ dữ liệu.
Không tự giảm batch size khi thiếu bộ nhớ. Muốn thay đổi cấu hình, chạy lại cell 1 rồi cell trọng số
và cell này. Kết quả CSV được lưu sau mỗi batch. Việc tải/cài đặt không nằm trong số đo inference.


In [ ]:
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
run_python("benchmark_sots_runner", RUNNER_SOURCE)


## 7. Xem bảng tổng hợp, từng batch và GFLOPs

FPS lấy từ forward gốc; cột stage phản ánh lượt đo phân rã riêng. Mili giây mỗi ảnh ở batch 16
là chi phí chia theo thông lượng, không phải latency batch 1. Padding vào cùng kích thước mỗi batch
được ghi lại để so sánh FLOPs đúng với kích thước thực xử lý.


In [ ]:
import csv
from IPython.display import display, HTML
import html

def show_csv(path, limit=None):
    if not path.is_file():
        print("Chưa có kết quả:", path)
        return
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        columns = reader.fieldnames
        rows = list(reader)
    if limit is not None:
        rows = rows[:limit]
    table = "<table><thead><tr>" + "".join(
        f"<th>{html.escape(c)}</th>" for c in columns) + "</tr></thead><tbody>"
    for row in rows:
        table += "<tr>" + "".join(f"<td>{html.escape(row[c])}</td>" for c in columns) + "</tr>"
    display(HTML(table + "</tbody></table>"))

show_csv(OUTPUT_ROOT / "summary.csv")
show_csv(OUTPUT_ROOT / "stage_gflops.csv")
for label in ("SOTS-ITS", "SOTS-OTS"):
    print(label, "— 5 batch đầu:")
    show_csv(OUTPUT_ROOT / f"{label}_batches.csv", limit=5)
print("Metadata:")
if (OUTPUT_ROOT / "metadata.json").exists():
    print((OUTPUT_ROOT / "metadata.json").read_text(encoding="utf-8"))


## 8. Tải kết quả về máy

ZIP chứa bảng tổng hợp, các CSV từng ảnh/từng batch, GFLOPs và thông tin GPU/cấu hình.
Mã benchmark sinh ra nằm trong thư mục làm việc; không sửa implementation WDMamba.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(WORK_ROOT / "wdmamba_sots_results"), "zip", OUTPUT_ROOT)
files.download(archive)


## Lưu ý khi so sánh

- `MAX_SIDE = 512` có thể làm thay đổi PSNR/SSIM so với độ phân giải gốc. Muốn báo cáo điểm gốc,
  đặt 0 và dùng GPU đủ VRAM cho batch 16.
- Các phép đếm bổ sung cho FFT/scan là xấp xỉ, không suy ra FPS từ GFLOPs.
- CSV batch có thời gian chờ DataLoader, truyền CPU → GPU và đưa output về CPU. Không dùng
  các cột này để gọi là FPS đầu-cuối vì còn lượt đo lặp và lượt phân rã riêng.
- Kết quả có thể khác benchmark dùng AMP, TF32 hoặc backend khác. Notebook mặc định FP32,
  tắt TF32 để dễ đối chiếu chất lượng.
- Tải kết quả trước khi kết thúc Colab; dữ liệu dưới `/content` không tồn tại sau khi runtime bị xóa.
